# **EDA 시각화 마스터**(EDA · Visualization Mastery)

## 2차시 (3시간) — 어떤 상황에서 어떤 그래프가 가장 적합한가?

본 차시는 다음 네 영역의 시각화 도구를 체계적으로 다룬다.

| 영역 | 핵심 질문 | 대표 도구 |
|---|---|---|
| **Part 1**(분포) | 한 변수가 어떻게 퍼져 있는가? | `histplot`·`kdeplot`·`boxplot`·`violinplot` |
| **Part 2**(관계) | 두 변수는 어떤 관계인가? | `scatterplot`·`regplot`·`pairplot`·`heatmap` |
| **Part 3**(비교) | 그룹 간 어떻게 다른가? | `barplot`·`countplot`·`grouped bar` |
| **Part 4**(인터랙티브·고차원) | 어떻게 인터랙티브하게 보는가? | `plotly`·`PCA`·`t-SNE` |

---

> **시각화의 제1원칙**: "항상 그래프를 먼저 그려라."
>
> 통계량이 같아도 데이터의 실제 모습은 완전히 다를 수 있다 — **앤스콤의 사중주**(Anscombe's Quartet)가 1973년 보여준 사실이다. 본 차시는 이 원칙에서 출발한다.


## **0. 환경 설정**(Environment Setup)

다음 셀을 그대로 실행한다. 한글 폰트, **색맹 안전 팔레트**(Colorblind-Safe Palette), 고화질 출력을 한 번에 설정한다.

In [ ]:
# 기본 라이브러리
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# 한글 폰트 (Colab 환경)
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# ★ 색맹 안전 팔레트 (Colorblind-Safe)
sns.set_palette("colorblind")
sns.set_style("whitegrid")

print('환경 설정 완료')
print(f'  pandas {pd.__version__}')
print(f'  seaborn {sns.__version__}')
print('  팔레트: colorblind (적록 색맹 안전)')

### **색맹 안전 팔레트**(Colorblind-Safe)는 왜 중요한가?

전 세계 **남성의 약 8%**, **여성의 약 0.5%**가 색각 이상을 가진다. 30명 반에 남학생 15명이면 통계적으로 약 **1.2명**이 영향을 받는다.

| 사용 권장 | 사용 자제 |
|---|---|
| `colorblind`(범주형 8색) | `red+green` 조합 |
| `viridis`(연속값) | `jet`·`rainbow` |
| `cividis`(색맹 전용 설계) | (적록 색맹에 동일하게 보임) |

> 데이터 시각화는 "예쁜 그래프"가 아니라 **"모든 사람이 읽을 수 있는 그래프"**를 만드는 것이다.
> 모든 노트북 첫 셀에 `sns.set_palette("colorblind")`을 두는 습관을 들이는 것이 좋다.

### **데이터 로드**(Data Loading)

| 역할 | 데이터셋 | 용도 |
|---|---|---|
| **Ping**(강사 시연) | Airbnb NYC 2019 | 가격 분포·지역 비교·관계 탐색 |
| **Pong**(학생 실습) | COVID-19 Compact | 시계열 분포·국가 비교 |

In [ ]:
# Ping 데이터 — Airbnb NYC 2019
AIRBNB_URL = ('https://raw.githubusercontent.com/leina99-lab/classes/main/'
              'AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AB_NYC_2019.csv')

try:
    airbnb = pd.read_csv(AIRBNB_URL)
    print(f'Airbnb 로드 성공: {airbnb.shape}')
except Exception:
    # 폴백 합성 데이터
    np.random.seed(42)
    n = 1000
    airbnb = pd.DataFrame({
        'neighbourhood_group': np.random.choice(
            ['Manhattan','Brooklyn','Queens','Bronx','Staten Island'],
            n, p=[0.45,0.40,0.10,0.03,0.02]),
        'room_type': np.random.choice(
            ['Entire home/apt','Private room','Shared room'],
            n, p=[0.52,0.45,0.03]),
        'price': np.random.lognormal(4.7, 0.7, n).astype(int) + 30,
        'minimum_nights': np.random.choice([1,2,3,5,7,30], n,
                                           p=[0.40,0.20,0.10,0.10,0.10,0.10]),
        'number_of_reviews': np.random.poisson(15, n),
        'reviews_per_month': np.round(np.random.exponential(1.0, n), 2),
        'calculated_host_listings_count': np.random.choice([1,2,3,5,10,50], n,
                                           p=[0.65,0.15,0.08,0.05,0.04,0.03]),
        'availability_365': np.random.randint(0, 366, n),
        'longitude': np.random.uniform(-74.05, -73.85, n),
        'latitude': np.random.uniform(40.65, 40.85, n),
    })
    print(f'폴백 데이터 사용: {airbnb.shape}')

# 분석용 정제
airbnb = airbnb[airbnb['price'].between(10, 1000)].copy()
print(f'정제 후: {airbnb.shape}')
airbnb.head(3)

In [ ]:
# Pong 데이터 — COVID-19
COVID_URL = ('https://raw.githubusercontent.com/leina99-lab/classes/main/'
             'AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/compact.csv')

try:
    covid = pd.read_csv(COVID_URL)
    if 'Date' in covid.columns:
        covid['Date'] = pd.to_datetime(covid['Date'])
    print(f'COVID 로드 성공: {covid.shape}')
except Exception:
    # 폴백 합성 데이터
    np.random.seed(7)
    countries = ['Korea','USA','Japan','Germany','Brazil','India','UK','France']
    dates = pd.date_range('2020-03-01', '2021-03-01', freq='D')
    rows = []
    for c in countries:
        base = np.random.uniform(50, 5000)
        wave = np.sin(np.linspace(0, 4*np.pi, len(dates))) + 1.2
        for i, d in enumerate(dates):
            confirmed = max(0, int(base * wave[i] * (1 + i/200) + np.random.randn()*100))
            deaths = int(confirmed * 0.018) + np.random.randint(0, 5)
            recovered = int(confirmed * 0.7) + np.random.randint(0, 50)
            rows.append([c, d, confirmed, deaths, recovered])
    covid = pd.DataFrame(rows,
                         columns=['Country/Region','Date','Confirmed','Deaths','Recovered'])
    print(f'폴백 데이터 사용: {covid.shape}')

covid.head(3)

---

## **0.1 앤스콤의 사중주**(Anscombe's Quartet) — 시각화의 제1원칙

1973년 통계학자 **Frank Anscombe**는 충격적 데이터를 발표하였다. 네 개의 데이터셋이 있는데 모든 통계량이 **완전히 동일**하다.

| 통계량 | I | II | III | IV |
|---|---|---|---|---|
| **mean(x)** | 9.0 | 9.0 | 9.0 | 9.0 |
| **mean(y)** | 7.50 | 7.50 | 7.50 | 7.50 |
| **std(x)** | 3.32 | 3.32 | 3.32 | 3.32 |
| **std(y)** | 2.03 | 2.03 | 2.03 | 2.03 |
| **corr(x,y)** | 0.816 | 0.816 | 0.816 | 0.816 |
| **회귀식** | y=3+0.5x | y=3+0.5x | y=3+0.5x | y=3+0.5x |

평균, 표준편차, 상관계수, 회귀식까지 전부 같다. 그러나 **그래프를 그리면 완전히 다르다**.

In [ ]:
# Seaborn 내장 데이터 사용
anscombe = sns.load_dataset('anscombe')
print(anscombe.groupby('dataset')[['x','y']].describe().round(2))

In [ ]:
# 시각화 — 4개 데이터셋의 산점도 + 회귀선
g = sns.lmplot(data=anscombe, x='x', y='y',
               col='dataset', col_wrap=2,
               height=3.5, aspect=1.2,
               scatter_kws={'s': 60, 'alpha': 0.8},
               palette='colorblind')
g.set_titles('데이터셋 {col_name}')
plt.suptitle("앤스콤의 사중주 — 통계량은 같지만 패턴은 완전히 다르다",
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### **결과 해석**(Interpretation)

- **데이터셋 I**: 교과서적 선형 관계 — 회귀선이 적절하다.
- **데이터셋 II**: 실제로는 **곡선**(포물선) 관계 — 직선 모델은 완전히 잘못된 것이다.
- **데이터셋 III**: 거의 완벽한 직선이지만 **이상치 1개**가 회귀선을 끌어당기고 있다.
- **데이터셋 IV**: x값이 거의 동일한데 한 점만 멀리 있어 **상관관계 자체가 무의미**하다.

> **앤스콤의 교훈**: 통계량(mean, std, corr)만으로는 데이터를 이해할 수 없다. **항상 그래프를 먼저 그려라**. 이것이 EDA의 제1원칙이다.

> **AI 연결**: ML 모델도 **손실 함수**(Loss) 값만 보면 속는다. 학습 곡선(Learning Curve), 잔차 플롯(Residual Plot)을 시각화해야 **과적합**·**데이터 누수**(Data Leakage) 같은 문제를 발견할 수 있다.

---

# **Part 1. 분포 시각화**(Distribution)

> **핵심 질문**: 한 변수의 값들이 **어떻게 퍼져 있는가**?

| 도구 | 무엇을 보는가 | 한계 |
|---|---|---|
| **`histplot`**(히스토그램) | 구간별 빈도 — 분포의 전반적 모양 | 구간 개수에 민감 |
| **`kdeplot`**(커널 밀도) | 부드러운 밀도 곡선 — 연속적 분포 | 봉우리 개수 왜곡 가능 |
| **`boxplot`**(박스플롯) | 중앙값·IQR·이상치를 한눈에 — 그룹 비교 최적 | 다봉분포를 놓침 |
| **`violinplot`**(바이올린) | 박스플롯 + 밀도 — **이봉분포**(bimodal) 탐지 | 표본이 작으면 부정확 |

---

## **1.1 히스토그램 + KDE — 한 변수의 전체 분포**

### Ping 1 — Airbnb 가격 분포

```python
sns.histplot(data=df, x='price', kde=True)
```

`kde=True`를 켜면 막대 위에 **커널 밀도**(Kernel Density Estimate) 곡선이 함께 그려진다. 막대는 구간 빈도, 곡선은 연속적 추정이다.

In [ ]:
# 가격 분포 — 평균, 중앙값을 함께 표시
plt.figure(figsize=(11, 5))
sns.histplot(data=airbnb, x='price', bins=50, kde=True)
plt.axvline(airbnb['price'].mean(), color='red', linestyle='--', lw=2,
            label=f"평균: ${airbnb['price'].mean():.0f}")
plt.axvline(airbnb['price'].median(), color='blue', linestyle='-', lw=2,
            label=f"중앙값: ${airbnb['price'].median():.0f}")
plt.legend(fontsize=11)
plt.title('Airbnb NYC 가격 분포 — 오른쪽으로 긴 꼬리', fontsize=13, fontweight='bold')
plt.xlabel('가격 ($)'); plt.ylabel('빈도')
plt.show()

### **결과 해석**

- 분포가 **오른쪽으로 길게**(right-skewed) 늘어져 있다 — 소수의 고가 숙소가 평균을 끌어올린다.
- **평균 > 중앙값** 관계는 오른쪽 꼬리 분포의 전형적 신호이다.
- 이런 데이터에서는 **중앙값이 더 신뢰할 수 있는 대표값**이다.

> **AI 연결**: 많은 ML 알고리즘(선형 회귀, KNN, SVM, 신경망)은 **정규분포 가정** 또는 **스케일 통일**을 요구한다. 왜도가 큰 분포는 `np.log1p()`(로그 변환), `StandardScaler`(표준화), `PowerTransformer`(Box-Cox/Yeo-Johnson) 등으로 변환한 뒤 모델에 입력한다.

### ✎ 개념 확인 1

다음 코드의 결과로 가장 적절한 설명은?

```python
sns.histplot(data=df, x='income', bins=30, kde=True)
plt.axvline(df['income'].mean(), color='red')
plt.axvline(df['income'].median(), color='blue')
```

**(A)** 평균(빨강)과 중앙값(파랑)이 같은 위치에 있다면 정규분포에 가깝다.
**(B)** 평균(빨강)이 중앙값(파랑)보다 오른쪽에 있다면 분포는 왼쪽으로 치우쳐 있다.
**(C)** `kde=True`는 데이터를 정규화하여 다시 그린다.

<details><summary>▶ 정답 보기</summary>

**(A)** 평균과 중앙값이 거의 일치하면 분포가 대칭적이다. **정규분포에 가까운 신호**이다.

(B)는 반대 방향이다. 평균 > 중앙값 → 분포는 **오른쪽**으로 긴 꼬리를 가진다.
(C)의 `kde`는 정규화가 아니라 부드러운 **밀도 추정 곡선**을 추가하는 것이다.
</details>

### Pong 1 — COVID-19 일별 신규 확진자 분포

학생 실습. 한국의 일별 신규 확진자 분포를 그리고, 평균/중앙값을 표시하라.

In [ ]:
# Pong 1 - 한국 일별 신규 확진자 분포
korea = covid[covid['Country/Region'] == 'Korea'].sort_values('Date').copy()
korea['NewCases'] = korea['Confirmed'].diff().clip(lower=0)
korea = korea.dropna(subset=['NewCases'])

plt.figure(figsize=(11, 5))
sns.histplot(data=korea, x='NewCases', bins=40, kde=True)
plt.axvline(korea['NewCases'].mean(), color='red', linestyle='--', lw=2,
            label=f"평균: {korea['NewCases'].mean():.0f}")
plt.axvline(korea['NewCases'].median(), color='blue', linestyle='-', lw=2,
            label=f"중앙값: {korea['NewCases'].median():.0f}")
plt.legend(); plt.title('한국 — 일별 신규 확진자 분포', fontsize=13, fontweight='bold')
plt.xlabel('일별 신규 확진자'); plt.ylabel('일수')
plt.show()

print(f'\n왜도(skew): {korea["NewCases"].skew():.2f}')

> **Pong 1 관찰**: 평균과 중앙값의 위치를 비교하라. 분포가 대칭인가, 한쪽으로 치우쳤는가? 왜도(skew) 값과 일치하는가?

---

## **1.2 박스플롯 — 그룹 간 분포 비교의 표준**

### **박스플롯 구조도**

```
   이상치   수염         상자            수염   이상치
     ○ ────┤     ┌──────┬──────┐     ├──── ○
           │     │      │      │     │
       Q1−1.5×IQR  Q1   Q2(중앙) Q3  Q3+1.5×IQR

   상자(Box)  = Q1~Q3 = 데이터의 **중간 50%**
   가운데 선  = Q2 = 중앙값
   수염       = Q1−1.5×IQR ~ Q3+1.5×IQR
   수염 바깥  = 이상치(Outlier)
```

박스플롯 **하나로 중앙값·퍼짐 정도·이상치를 동시에** 볼 수 있어 그룹 비교에 가장 효율적이다.

### Ping 2 — 자치구별 가격 박스플롯

```python
sns.boxplot(data=df, x='neighbourhood_group', y='price', palette='colorblind')
```

In [ ]:
# 자치구별 가격 — 중앙값 내림차순 정렬
order = (airbnb.groupby('neighbourhood_group')['price']
         .median().sort_values(ascending=False).index)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (A) 전체 박스플롯 — 이상치까지 모두 표시
sns.boxplot(data=airbnb, x='neighbourhood_group', y='price',
            order=order, palette='colorblind', ax=axes[0])
axes[0].set_title('(A) 전체 가격 — 이상치 포함', fontsize=12, fontweight='bold')
axes[0].set_xlabel('자치구'); axes[0].set_ylabel('가격 ($)')

# (B) $300 이하 확대
under300 = airbnb[airbnb['price'] <= 300]
sns.boxplot(data=under300, x='neighbourhood_group', y='price',
            order=order, palette='colorblind', ax=axes[1])
axes[1].set_title('(B) $300 이하 — 박스 내부가 잘 보인다', fontsize=12, fontweight='bold')
axes[1].set_xlabel('자치구'); axes[1].set_ylabel('가격 ($)')

plt.tight_layout(); plt.show()

print('\n자치구별 통계:')
print(airbnb.groupby('neighbourhood_group')['price']
      .agg(['median','mean','std','count']).round(0).sort_values('median', ascending=False))

### **박스플롯 읽는 순서 (3단계)**

1. **상자 위치**(Q2 위치)를 비교한다 → 어느 그룹의 중앙값이 가장 높은가?
2. **상자 길이**(IQR=Q3-Q1)를 비교한다 → 어느 그룹의 가격 편차가 큰가?
3. **이상치(○)** 의 분포를 본다 → 초고가 숙소의 위치는 어디인가?

> **AI 연결**: 박스플롯의 이상치 기준(Q1−1.5×IQR, Q3+1.5×IQR)은 **IQR 방법**(IQR Method)으로, 이상 탐지(Anomaly Detection)의 가장 기초적 룰이다. 더 정교한 방법으로는 **Isolation Forest**, **One-Class SVM**, **Autoencoder 기반 이상 탐지**가 있다.

### ✎ 개념 확인 2

박스플롯에서 **상자 안의 가운데 선**이 의미하는 것은?

**(A)** 평균(mean)
**(B)** 중앙값(median, Q2)
**(C)** 최빈값(mode)

<details><summary>▶ 정답 보기</summary>

**(B)** 중앙값(median, Q2). 박스플롯은 평균을 표시하지 않는다(필요하면 `showmeans=True` 옵션).

박스플롯이 사용하는 5수치(5-number summary)는 **min, Q1, Q2(중앙값), Q3, max**이다. 평균은 이상치에 민감하기 때문에 박스플롯의 핵심에서 제외된다.
</details>

### Pong 2 — 국가별 일별 신규 확진자 박스플롯

학생 실습. 5개 국가의 일별 신규 확진자 분포를 박스플롯으로 비교하라. 중앙값이 가장 높은 국가는?

In [ ]:
# Pong 2 - 국가별 신규 확진자 박스플롯
covid_sorted = covid.sort_values(['Country/Region','Date']).copy()
covid_sorted['NewCases'] = covid_sorted.groupby('Country/Region')['Confirmed'].diff().clip(lower=0)
covid_clean = covid_sorted.dropna(subset=['NewCases'])

plt.figure(figsize=(12, 6))
order_c = (covid_clean.groupby('Country/Region')['NewCases']
           .median().sort_values(ascending=False).index)
sns.boxplot(data=covid_clean, x='Country/Region', y='NewCases',
            order=order_c, palette='colorblind')
plt.title('국가별 일별 신규 확진자 분포 (박스플롯)', fontsize=13, fontweight='bold')
plt.xlabel('국가'); plt.ylabel('일별 신규 확진자')
plt.xticks(rotation=20)
plt.show()

---

## **1.3 바이올린 플롯 — 박스플롯이 놓치는 것**

박스플롯은 **이봉분포**(bimodal, 봉우리가 2개인 분포)를 잡아내지 못한다. 박스 길이만 봐서는 가운데가 비어 있는지 알 수 없기 때문이다.

```
박스플롯 (정보 부족)        바이올린 플롯 (분포 모양 보임)

  │     ┌────┬────┐ │       │     ╭──╮ ╭──╮ │
  │─────┤    │    ├─│       │  ───┤  ╲╱  ├──│   ← 봉우리 2개!
  │     └────┴────┘ │       │     ╰──╯╰──╯ │
```

### Ping 3 — Airbnb 자치구별 바이올린 플롯

In [ ]:
# 바이올린 플롯 - 분포 모양까지 보여준다
under300 = airbnb[airbnb['price'] <= 300].copy()
order = (under300.groupby('neighbourhood_group')['price']
         .median().sort_values(ascending=False).index)

plt.figure(figsize=(12, 6))
sns.violinplot(data=under300, x='neighbourhood_group', y='price',
               order=order, palette='colorblind',
               inner='quartile',  # 사분위선 표시
               cut=0)             # 데이터 범위 밖 곡선 제거
plt.title('자치구별 가격 바이올린 ($300 이하) — 봉우리 모양 관찰', fontsize=13, fontweight='bold')
plt.xlabel('자치구'); plt.ylabel('가격 ($)')
plt.show()

### **바이올린 읽는 법**

- **넓은 부분** = 그 가격대에 숙소가 많다.
- **봉우리(peak)가 1개** = 단봉분포(unimodal).
- **봉우리가 2개** = **이봉분포**(bimodal) → 가격대가 두 그룹으로 나뉨을 시사한다.
- `inner='quartile'`로 박스플롯의 사분위선을 함께 표시한다.

> **AI 연결**: 클러스터링(K-means 등) 알고리즘은 데이터가 여러 군집으로 나뉘어 있을 때만 의미가 있다. 바이올린 플롯에서 **이봉분포**가 발견되면 클러스터링·혼합 모델(Gaussian Mixture)을 적용하여 군집을 분리할 수 있다.

### Pong 3 — 국가별 신규 확진자 바이올린

학생 실습. 박스플롯에서 보이지 않던 분포 모양이 바이올린에서는 어떻게 드러나는가?

In [ ]:
# Pong 3 - 국가별 신규 확진자 바이올린
plt.figure(figsize=(12, 6))
sns.violinplot(data=covid_clean, x='Country/Region', y='NewCases',
               order=order_c, palette='colorblind',
               inner='quartile', cut=0)
plt.title('국가별 일별 신규 확진자 바이올린', fontsize=13, fontweight='bold')
plt.xlabel('국가'); plt.ylabel('일별 신규 확진자')
plt.xticks(rotation=20)
plt.show()

---

## **1.4 분포-ML 연결 — 왜도, 로그 변환, 표준화**

> 많은 ML 알고리즘은 **정규분포 가정** 또는 **스케일 통일**을 요구한다. EDA에서 발견한 왜도(skew)는 곧 전처리 의사결정으로 이어진다.

| 알고리즘 | 가정 | 위반 시 문제 |
|---|---|---|
| 선형 회귀 | 잔차가 정규분포 | 신뢰구간·p-value 부정확 |
| KNN, SVM | 특성 스케일 유사 | 큰 스케일 특성이 거리 지배 |
| 신경망·LLM | 입력이 정규화됨 | 학습 속도 급감, **기울기 폭발** |

> **GPT/BERT의 Layer Normalization**도 같은 원리이다. 각 층의 출력을 평균=0, 분산=1로 정규화한다 — 본질적으로 `StandardScaler`와 동일하다.

In [ ]:
# 왜도(Skew) → 변환 → 정규화 시각화
from scipy.stats import skew

# Airbnb 가격에 적용
price = airbnb['price'].values
log_price = np.log1p(price)
from sklearn.preprocessing import StandardScaler
scaled_price = StandardScaler().fit_transform(price.reshape(-1, 1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (A) 원본
axes[0].hist(price, bins=50, color='#3b82f6', alpha=0.6, edgecolor='white')
axes[0].set_title(f'(A) 원본 — skew = {skew(price):.2f}',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('price ($)'); axes[0].set_ylabel('빈도')

# (B) 로그 변환
axes[1].hist(log_price, bins=50, color='#10b981', alpha=0.6, edgecolor='white')
axes[1].set_title(f'(B) 로그 변환 — skew = {skew(log_price):.2f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(1 + price)')

# (C) 표준화
axes[2].hist(scaled_price, bins=50, color='#f59e0b', alpha=0.6, edgecolor='white')
axes[2].set_title(f'(C) StandardScaler — μ≈0, σ≈1',
                  fontsize=12, fontweight='bold')
axes[2].set_xlabel('z-score')

plt.suptitle('분포 변환 — ML 입력으로 적합한 형태로', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f'원본 왜도:        {skew(price):.3f}')
print(f'로그 변환 후 왜도: {skew(log_price):.3f}')

### **결과 해석**

- 원본은 오른쪽으로 길게 늘어진 강한 양의 왜도(skew ≫ 1)를 보인다.
- **로그 변환**(`np.log1p`)을 적용하면 분포가 대칭에 가까워진다 — 정규분포 가정 알고리즘의 입력으로 적합해진다.
- **표준화**(`StandardScaler`)는 평균을 0, 표준편차를 1로 맞추어 **스케일을 통일**한다.

> **실무 규칙**: |skew| > 1이면 변환을 고려한다. 변환 후에도 |skew| > 1이면 다른 변환(Box-Cox, Yeo-Johnson)을 시도한다.

### ✎ 개념 확인 3

`skew = 2.5`인 변수를 KNN 분류기에 그대로 입력하였다. 가장 적절한 처치는?

**(A)** 그대로 사용한다 — KNN은 분포에 무관하다.
**(B)** `np.log1p()`로 로그 변환한 뒤 `StandardScaler`로 표준화한다.
**(C)** `MinMaxScaler`로 0~1 사이로 줄이기만 하면 된다.

<details><summary>▶ 정답 보기</summary>

**(B)** 강한 양의 왜도(skew=2.5)는 먼저 **로그 변환**으로 대칭화하고, 그 뒤 **표준화**로 다른 특성과 스케일을 맞추는 것이 정석이다.

KNN은 거리 기반이므로 스케일과 분포에 모두 민감하다. (A)는 거리 계산이 한쪽으로 쏠리고, (C)는 왜도를 그대로 두므로 거리 분포가 왜곡된다.
</details>

---

## **Part 1 연습문제** — 분포 시각화 10문항

각 문항에는 토글 정답이 함께 제공된다. 직접 풀어본 뒤 확인하라.

---

### 문제 1. 평균 vs 중앙값

오른쪽 꼬리가 긴 분포에서 평균과 중앙값의 관계로 옳은 것은?

**(A)** 평균 > 중앙값 &nbsp;&nbsp; **(B)** 평균 = 중앙값 &nbsp;&nbsp; **(C)** 평균 < 중앙값

<details><summary>▶ 정답</summary>(A) 평균 > 중앙값. 오른쪽 꼬리의 큰 값들이 평균을 끌어올린다.</details>

---

### 문제 2. KDE 옵션

`sns.histplot(data=df, x='price', kde=True)`에서 `kde=True`의 효과는?

<details><summary>▶ 정답</summary>막대 위에 **커널 밀도 추정**(Kernel Density Estimation) 곡선을 추가한다. 데이터의 부드러운 분포 모양을 함께 보여준다.</details>

---

### 문제 3. 박스플롯 5수치

박스플롯이 표시하는 5수치를 모두 쓰시오.

<details><summary>▶ 정답</summary>**최솟값, Q1(25%), Q2(중앙값, 50%), Q3(75%), 최댓값** (단, 수염은 Q1−1.5×IQR ~ Q3+1.5×IQR로 제한되며 그 바깥은 이상치로 표시).</details>

---

### 문제 4. IQR 정의

IQR(사분위 범위)의 정의를 쓰시오.

<details><summary>▶ 정답</summary>`IQR = Q3 − Q1`. 데이터의 **중간 50%**가 차지하는 범위. 박스플롯의 박스 길이에 해당한다.</details>

---

### 문제 5. 이상치 기준

박스플롯이 이상치로 표시하는 데이터의 조건은?

<details><summary>▶ 정답</summary>`x < Q1 − 1.5 × IQR` 또는 `x > Q3 + 1.5 × IQR`. 이를 **IQR 방법**(Tukey's fences)이라 한다.</details>

---

### 문제 6. 바이올린 vs 박스플롯

바이올린 플롯이 박스플롯보다 우월한 상황은?

<details><summary>▶ 정답</summary>**이봉분포**(bimodal) 또는 다봉분포 탐지. 박스플롯은 박스 길이만 보이지만, 바이올린은 분포 모양 자체를 보여준다.</details>

---

### 문제 7. 색맹 안전 팔레트

다음 중 색맹 안전 팔레트가 **아닌** 것은?

**(A)** colorblind &nbsp;&nbsp; **(B)** viridis &nbsp;&nbsp; **(C)** jet &nbsp;&nbsp; **(D)** cividis

<details><summary>▶ 정답</summary>**(C) jet**. jet/rainbow 팔레트는 적록 색맹에게 중간 구간이 구분되지 않는다.</details>

---

### 문제 8. 왜도와 변환

|skew| > 1인 분포에 가장 자주 적용되는 변환 함수 두 가지를 쓰시오.

<details><summary>▶ 정답</summary>`np.log1p(x)`(로그 변환) 또는 `PowerTransformer`(Box-Cox/Yeo-Johnson). 음수가 포함된 경우는 Yeo-Johnson을 사용한다.</details>

---

### 문제 9. 코드 빈칸

```python
# 빈칸 채우기 - 자치구별 박스플롯, 색맹 안전
sns._______(data=airbnb, x=_______, y='price',
            palette=_______)
```

<details><summary>▶ 정답</summary>

```python
sns.boxplot(data=airbnb, x='neighbourhood_group', y='price',
            palette='colorblind')
```
</details>

---

### 문제 10. AI 연결 — Layer Normalization

GPT/BERT의 **Layer Normalization**이 EDA의 어떤 개념과 본질적으로 같은가?

<details><summary>▶ 정답</summary>**StandardScaler에 의한 표준화**. 각 층의 출력을 평균=0, 분산=1로 정규화한다는 점에서 동일하다. "분포를 정규화하면 학습이 안정된다"는 원칙은 전통 ML부터 최신 LLM까지 관통한다.</details>

---

> **Part 1 마무리**: 분포 시각화는 **데이터의 모양**을 묻는 것이다. 모양을 알아야 통계 가정의 충족 여부를 판단하고, 이상치를 처리하며, 적절한 ML 변환을 결정할 수 있다.

---

# **Part 2. 관계 시각화**(Relationship)

> **핵심 질문**: 두 변수는 **어떤 관계**인가?

| 도구 | 무엇을 보는가 | 한계 |
|---|---|---|
| **`scatterplot`**(산점도) | 두 변수의 분포 패턴 | 점이 많으면 겹침 |
| **`regplot`**(회귀선) | 산점도 + 직선 추세 | 비선형 관계 놓침 |
| **`pairplot`**(페어플롯) | 모든 변수 쌍 한눈에 | 변수 많으면 느림 |
| **`heatmap`**(히트맵) | 상관 행렬 시각화 | 비선형 관계 놓침 |

---

> **상관관계의 3가지 실전 목적**:
> 1. **특성 선택**(Feature Selection) — 타겟과 상관 높은 특성 선택
> 2. **다중공선성**(Multicollinearity) 탐지 — 특성 간 |r| > 0.9 → 하나 제거
> 3. **데이터 누수**(Data Leakage) 탐지 — 타겟과 |r| ≈ 1.0 → 미래 정보 누출 차단

## **2.1 산점도 — 두 변수 관계의 출발점**

```python
sns.scatterplot(data=df, x='x_var', y='y_var')
```

산점도의 점 하나는 **데이터 한 행**(한 관측 단위)이다.

### Ping 4 — 리뷰 수 vs 가격 (지역별 색상)

In [ ]:
# 산점도 — hue + style로 색맹 안전 + 모양 분리
under300 = airbnb[airbnb['price'] <= 300].copy()

plt.figure(figsize=(11, 7))
sns.scatterplot(data=under300,
                x='number_of_reviews', y='price',
                hue='neighbourhood_group',
                style='neighbourhood_group',  # ★ 색상 + 모양으로 이중 구분
                palette='colorblind',
                alpha=0.5, s=30)
plt.title('리뷰 수 vs 가격 — 색상과 모양으로 자치구 분리', fontsize=13, fontweight='bold')
plt.xlabel('리뷰 수'); plt.ylabel('가격 ($)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

### **결과 해석**

- 점 하나 = 한 숙소이다.
- `hue=`로 색상을 분리하고, `style=`로 마커 모양도 함께 분리하면 **흑백 인쇄 시에도 그룹 구분**이 가능하다.
- 리뷰 수가 많은 숙소가 비싸거나 싸다는 뚜렷한 선형 관계는 보이지 않는다.

> **상관 ≠ 인과**:  "리뷰가 많아서 가격이 비싸다" 같은 인과 표현은 사용하지 않는다. **"~와 관련이 있다"**(상관) vs **"~때문이다"**(인과)를 엄격히 구분하라.

> **고전 사례**: 아이스크림 판매량과 익사 사고는 r ≈ +0.8의 상관을 가진다. 그러나 아이스크림이 익사를 일으킨 것이 아니라, **여름(기온)**이라는 숨은 변수가 둘 다에 영향을 준 것이다.

### ✎ 개념 확인 4

`sns.scatterplot()`에서 `hue=`와 `style=`을 **동시에** 사용하는 이유로 가장 적절한 것은?

**(A)** 데이터를 더 빠르게 그릴 수 있다.
**(B)** 색맹인 사람과 흑백 인쇄 환경에서도 그룹 구분이 가능하다.
**(C)** 점의 크기를 자동으로 조정한다.

<details><summary>▶ 정답 보기</summary>

**(B)** 색상에만 의존하면 적록 색맹에게 동일하게 보일 수 있고, 흑백 인쇄 시 구분이 사라진다. **`style=`**(마커 모양)을 함께 사용하면 색상이 사라져도 그룹을 구분할 수 있다.
</details>

### Pong 4 — 확진자 수 vs 사망자 수 (국가별)

학생 실습. COVID-19 데이터에서 두 변수의 관계를 산점도로 그려라.

In [ ]:
# Pong 4 - 확진자 vs 사망자 산점도
plt.figure(figsize=(11, 7))
sns.scatterplot(data=covid_clean.sample(min(2000, len(covid_clean)), random_state=42),
                x='Confirmed', y='Deaths',
                hue='Country/Region', style='Country/Region',
                palette='colorblind', alpha=0.5, s=25)
plt.title('확진자 누적 vs 사망자 누적 — 국가별', fontsize=13, fontweight='bold')
plt.xlabel('누적 확진자 수'); plt.ylabel('누적 사망자 수')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

---

## **2.2 회귀선이 들어간 산점도 — `regplot`**

```python
sns.regplot(data=df, x='x', y='y',
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
```

회귀선(regression line)은 두 변수의 **선형 추세**를 빨간 직선으로 보여준다. 신뢰구간(95%)도 함께 표시된다.

### Ping 5 — 가격과 가장 상관 높은 변수의 회귀선

In [ ]:
# 가격과의 상관계수 산출
num_cols = ['price','minimum_nights','number_of_reviews',
            'reviews_per_month','calculated_host_listings_count','availability_365']
corr_full = airbnb[num_cols].corr()
price_corr = corr_full['price'].drop('price').abs().sort_values(ascending=False)
print('가격과의 절대 상관계수 (내림차순):')
print(price_corr.round(3))

top2 = price_corr.index[:2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(top2):
    sns.regplot(data=under300, x=col, y='price',
                scatter_kws={'alpha':0.15, 's':10},
                line_kws={'color':'crimson', 'lw':2},
                ax=axes[i])
    r = under300[[col, 'price']].corr().iloc[0,1]
    axes[i].set_title(f'price vs {col} (r={r:.3f})',
                       fontsize=12, fontweight='bold')

plt.tight_layout(); plt.show()

### **결과 해석**

- 빨간 직선은 **최소제곱 회귀선**(OLS line)이다. 회귀식은 `y = ax + b`.
- 음영 영역은 **신뢰구간**(기본 95%)이다.
- 상관계수의 절대값이 0.3 미만이면 **약한 상관**이다 — 직선이 거의 수평에 가깝다.

> **상관계수 해석 가이드**:
> | \|r\| | 해석 |
> |---|---|
> | < 0.3 | 약한 상관 |
> | 0.3 ~ 0.7 | 보통 상관 |
> | > 0.7 | 강한 상관 |

---

## **2.3 페어플롯 — 모든 변수 쌍을 한 번에**

```python
sns.pairplot(df, hue='category', palette='colorblind')
```

`pairplot`은 N개 변수에 대해 N×N 격자를 만들어, 대각선에는 **각 변수의 분포**(KDE/히스토그램), 비대각선에는 **두 변수의 산점도**를 배치한다.

### Ping 6 — 4개 숫자 변수의 페어플롯

In [ ]:
# 페어플롯 - 시간이 다소 걸림
cols_pair = ['price','number_of_reviews','minimum_nights','availability_365']
df_pair = under300[cols_pair + ['neighbourhood_group']].sample(
    min(2000, len(under300)), random_state=42).dropna()

g = sns.pairplot(df_pair, hue='neighbourhood_group',
                 palette='colorblind',
                 plot_kws={'alpha':0.3, 's':12},
                 diag_kind='kde',
                 height=2.3)
g.fig.suptitle('주요 숫자 변수 페어플롯 — 자치구별', y=1.01, fontsize=13, fontweight='bold')
plt.show()

### **페어플롯 읽는 법**

- **대각선**: 각 변수의 KDE — 분포를 본다.
- **비대각선**: 두 변수의 산점도 — 관계를 본다.
- 대각선 위와 아래는 **거울상**이므로 한쪽만 보면 된다(상삼각 또는 하삼각).

> 변수가 10개 이상이면 페어플롯은 너무 커진다. **상관 히트맵**으로 먼저 좁히고 페어플롯을 적용하는 워크플로우가 효율적이다.

---

## **2.4 상관 히트맵 — 다변수 관계 한눈에**

```python
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
```

**상관계수 행렬**을 색상 강도로 표현한다. `coolwarm`은 빨강-파랑 조합이므로 **적록 색맹에 안전**하다.

### Ping 7 — 숫자 변수 상관 히트맵

In [ ]:
# 상관 히트맵 — 삼각형 마스크 적용
corr = airbnb[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # 상삼각만 가림

plt.figure(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0,
            vmin=-1, vmax=1,
            linewidths=0.5, square=True,
            cbar_kws={'shrink':0.8, 'label':'상관계수'})
plt.title('Airbnb 숫자 변수 상관 히트맵 (하삼각)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### **결과 해석**

- 셀의 **숫자**는 두 변수의 **피어슨 상관계수**(Pearson correlation), -1과 +1 사이.
- 대각선은 자기 자신과의 상관 = 1.
- **빨강**(따뜻한 색) = 양의 상관, **파랑**(차가운 색) = 음의 상관, **흰색** = 무상관.
- 마스크(`np.triu`)로 상삼각을 가려 정보가 두 번 나오는 것을 방지한다.

### ✎ 개념 확인 5

상관 히트맵에서 두 특성 간 |r| = 0.95가 발견되었다. 이는 무엇을 시사하는가?

**(A)** 두 특성 모두 모델에 넣어야 한다 — 정보가 풍부하다.
**(B)** 다중공선성(Multicollinearity)이 있으므로 하나만 사용한다.
**(C)** 두 특성을 곱한 새 특성을 추가한다.

<details><summary>▶ 정답 보기</summary>

**(B)** 두 특성이 거의 같은 정보를 담고 있다. 둘 다 넣으면 **회귀 계수의 분산이 폭발**(coefficient instability)하고 해석이 어려워진다. 더 직관적인 하나만 남기거나 PCA 등으로 차원을 축소한다.

(A)는 정보 중복으로 모델이 불안정해진다. (C)는 다중공선성을 더 악화시킨다.
</details>

### Pong 5 — COVID 변수 상관 히트맵

학생 실습. COVID 데이터의 누적 변수들 간 상관을 히트맵으로 보아라.

In [ ]:
# Pong 5 - COVID 변수 상관 히트맵
covid_num = covid[['Confirmed','Deaths','Recovered']].dropna()
corr_c = covid_num.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_c, annot=True, fmt='.3f',
            cmap='coolwarm', center=0,
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title('COVID 누적 변수 상관 히트맵', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---

## **2.5 AI 연결 — 상관 히트맵으로 데이터 누수 탐지**

> **데이터 누수**(Data Leakage)는 ML 실무에서 **가장 위험한 실수**이다. 학습 시점에 알 수 없는 미래 정보가 입력 특성에 섞여 들어가는 현상이다. Kaggle 경쟁의 상위 솔루션이 무효화된 사례도 다수 있다.

### **상관 히트맵의 위험 신호 3가지**

| 발견 | 의심 | 조치 |
|---|---|---|
| 타겟과 \|r\| ≈ 1.0 | **데이터 누수** | 그 특성은 **학습에서 제외** |
| 두 특성 간 \|r\| > 0.9 | **다중공선성** | 하나만 남기거나 PCA |
| 모든 \|r\| < 0.1 | 선형 관계 부족 | **비선형 모델**(트리, NN) 고려 |

In [ ]:
# 데이터 누수 시나리오 시뮬레이션
np.random.seed(42)
n = 300
feature_1 = np.random.normal(50, 10, n)
feature_2 = 0.85 * feature_1 + np.random.normal(0, 5, n)  # 다중공선성!
feature_3 = np.random.normal(30, 8, n)
noise = np.random.normal(0, 3, n)
target = 0.6 * feature_1 + 0.3 * feature_3 + noise         # 진짜 관계
leaky = target * 0.95 + np.random.normal(0, 1, n)          # ★ 데이터 누수!

df_leak = pd.DataFrame({
    'feature_1': feature_1, 'feature_2': feature_2,
    'feature_3': feature_3, 'leaky_feat': leaky, 'target': target,
})

corr_leak = df_leak.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_leak, dtype=bool))
sns.heatmap(corr_leak, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('상관 히트맵 — 위험 신호 탐지', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('━━━ 발견된 위험 신호 ━━━')
print(f'[데이터 누수 의심] leaky_feat ↔ target = {corr_leak.loc["leaky_feat","target"]:.3f} (≈ 1.0)')
print(f'[다중공선성]      feature_1 ↔ feature_2 = {corr_leak.loc["feature_1","feature_2"]:.3f} (> 0.9)')

### **결과 해석**

- `leaky_feat ↔ target = 0.95` — **거의 완벽한 상관**. 학습 시점에 알 수 없는 미래 정보가 들어 있을 가능성이 매우 높다 → **모델에서 제외**.
- `feature_1 ↔ feature_2 = 0.86` — 다중공선성 → 둘 중 하나만 사용.

> 모델 학습 전에 상관 히트맵을 그리는 것은 **30초 투자로 30시간을 절약**하는 행동이다.

---

## **Part 2 연습문제** — 관계 시각화 10문항

---

### 문제 1. 산점도의 의미

산점도(scatter plot)에서 점 하나가 의미하는 것은?

<details><summary>▶ 정답</summary>**한 행의 데이터**(한 관측 단위). 한 점은 두 변수에 대한 **한 사람/한 숙소/한 측정값**의 좌표이다.</details>

---

### 문제 2. 상관 ≠ 인과

다음 중 **잘못된** 표현은?

**(A)** 광고비와 매출은 강한 양의 상관을 보인다.
**(B)** 광고비를 늘렸기 **때문에** 매출이 증가했다.
**(C)** 두 변수의 상관계수는 0.85로 강한 관련이 있다.

<details><summary>▶ 정답</summary>**(B)**. 상관은 **인과를 함의하지 않는다**. "~때문에"라는 인과 표현은 통제된 실험(A/B 테스트, RCT) 또는 인과 추론 분석(causal inference) 없이 사용해서는 안 된다.</details>

---

### 문제 3. regplot의 신뢰구간

`sns.regplot()`의 음영 영역이 의미하는 것은?

<details><summary>▶ 정답</summary>회귀선의 **95% 신뢰구간**(confidence interval). 이 영역만큼 회귀선이 흔들릴 수 있다는 의미. `ci=99`로 변경하거나 `ci=None`으로 끌 수 있다.</details>

---

### 문제 4. 페어플롯 대각선

`sns.pairplot()`의 대각선에 표시되는 것은?

<details><summary>▶ 정답</summary>**각 변수의 단변량 분포**. 기본은 히스토그램, `diag_kind='kde'`로 KDE로 변경 가능. 대각선 외 셀은 두 변수의 산점도이다.</details>

---

### 문제 5. 히트맵 cmap

상관 히트맵에 가장 적합한 컬러맵 두 개를 쓰시오.

<details><summary>▶ 정답</summary>**`coolwarm`** 또는 **`RdBu_r`**(빨강-파랑 계열). 0을 기준으로 양/음의 발산 컬러맵(diverging colormap)이며 적록 색맹에 안전하다. `center=0`을 함께 지정한다.</details>

---

### 문제 6. 마스크 효과

`mask = np.triu(np.ones_like(corr, dtype=bool))`을 히트맵에 적용하는 이유는?

<details><summary>▶ 정답</summary>상관 행렬은 **대칭**(symmetric)이므로 상삼각과 하삼각의 정보가 동일하다. 한쪽을 가려서 **시각적 잡음을 줄이고** 정보를 한 번만 보여준다.</details>

---

### 문제 7. 다중공선성 탐지

다중공선성이 의심되는 두 특성의 일반적 임계 |r|은?

<details><summary>▶ 정답</summary>**\|r\| > 0.9** (보수적), **\|r\| > 0.7~0.8** (실무적). 임계 위에서는 둘 중 하나를 제거하거나 PCA·정규화 회귀(Ridge/Lasso)를 사용한다.</details>

---

### 문제 8. 데이터 누수 신호

타겟과 |r| = 0.99를 보이는 특성을 발견하였다. 어떤 가능성을 의심해야 하는가?

<details><summary>▶ 정답</summary>**데이터 누수**(Data Leakage). 학습 시점에 알 수 없어야 할 미래 정보가 특성에 섞여 들어갔을 가능성. 예: "환자 진단" 데이터에 "처방받은 약" 컬럼이 입력 특성으로 포함된 경우.</details>

---

### 문제 9. 코드 빈칸

```python
# 빈칸 채우기 - 회귀선이 있는 산점도
sns._______(data=df, x='reviews', y='price',
            scatter_kws={'alpha':0.3},
            line_kws={'color':'_______'})
```

<details><summary>▶ 정답</summary>

```python
sns.regplot(data=df, x='reviews', y='price',
            scatter_kws={'alpha':0.3},
            line_kws={'color':'red'})
```
</details>

---

### 문제 10. AI 연결 — 차원 축소의 출발점

상관 히트맵이 차원 축소(PCA, Autoencoder) 적용의 **출발점**인 이유는?

<details><summary>▶ 정답</summary>높은 상관을 가진 특성들은 **같은 정보**를 담고 있으므로 압축이 가능함을 시사한다. PCA는 분산이 최대인 축으로 데이터를 회전·투영하여 상관된 특성을 통합한다. 상관이 거의 없는 특성들은 이미 독립적이므로 차원 축소의 효과가 작다.</details>

---

> **Part 2 마무리**: 관계 시각화는 **변수 간의 다리**를 보여준다. 그 다리가 진짜 다리인지(인과), 우연한 동행인지(상관), 함정인지(데이터 누수)를 분별하는 것이 분석가의 핵심 역량이다.

---

# **Part 3. 비교 시각화**(Comparison)

> **핵심 질문**: 그룹 간에 **어떻게 다른가**?

| 도구 | 무엇을 보는가 | 한계 |
|---|---|---|
| **`countplot`**(카운트) | 범주별 빈도 | 비율 정보 없음 |
| **`barplot`**(막대) | 범주별 통계량(평균 등) | 분포 정보 손실 |
| **그룹 막대**(grouped) | 두 범주의 교차 빈도 | 카테고리 많으면 복잡 |
| **`stripplot`**·**`swarmplot`** | 개별 점 + 그룹 비교 | 데이터 많으면 느림 |

---

## **3.1 카운트 플롯 — 범주별 빈도**

```python
sns.countplot(data=df, x='category', palette='colorblind')
```

`countplot`은 범주형 변수의 **각 값이 몇 번 등장하는지** 막대로 표시한다.

### Ping 8 — 자치구 × 숙소 유형 카운트

In [ ]:
# 카운트 플롯 - 매물 수가 많은 자치구 순으로 정렬
order_count = airbnb['neighbourhood_group'].value_counts().index

plt.figure(figsize=(11, 6))
sns.countplot(data=airbnb, x='neighbourhood_group',
              hue='room_type',
              order=order_count,
              palette='colorblind')
plt.title('자치구 × 숙소 유형별 매물 수', fontsize=13, fontweight='bold')
plt.xlabel('자치구'); plt.ylabel('매물 수')
plt.legend(title='Room Type', bbox_to_anchor=(1.02, 1))
plt.tight_layout(); plt.show()

### **결과 해석**

- 매물 수가 가장 많은 두 자치구는 Manhattan과 Brooklyn이다.
- Manhattan은 **Entire home/apt** 비율이, Brooklyn은 **Private room** 비율이 상대적으로 높다.
- `Shared room`은 모든 자치구에서 매물이 매우 적다.

> **AI 연결 — 클래스 불균형**: 분류 문제에서 클래스별 빈도 차이가 크면(예: 99:1) 모델은 다수 클래스만 예측하면서 높은 정확도를 얻는 함정에 빠진다. **`countplot`이 가장 먼저 발견해야 할 문제**이다. 대책으로 SMOTE(오버샘플링), 언더샘플링, `class_weight='balanced'` 등이 있다.

### ✎ 개념 확인 6

이진 분류 데이터에서 `countplot`을 그려보니 **0:1 = 95:5**의 비율이 나왔다. 가장 적절한 조치는?

**(A)** 데이터가 충분하므로 그대로 학습한다.
**(B)** SMOTE 오버샘플링·언더샘플링·`class_weight='balanced'` 등 불균형 처리를 한다.
**(C)** 다수 클래스(0)만 학습 데이터로 사용한다.

<details><summary>▶ 정답 보기</summary>

**(B)** 95:5는 심각한 클래스 불균형이다. 그대로 학습하면 모델이 "전부 0"이라고만 예측해도 정확도 95%를 달성하지만, **소수 클래스(1)에 대한 재현율(recall)은 0**이 된다.

핵심 대응:
- **SMOTE**(Synthetic Minority Oversampling) — 소수 클래스를 합성으로 늘림
- **`class_weight='balanced'`** — 손실 함수에서 소수 클래스에 큰 가중치
- **F1, ROC-AUC** 등 불균형에 강건한 지표 사용
</details>

### Pong 6 — COVID 데이터 국가별 관측 일수

학생 실습. 국가별로 관측 일수(데이터 행 수)를 카운트로 비교하라.

In [ ]:
# Pong 6 - 국가별 관측 일수
plt.figure(figsize=(11, 5))
order_cv = covid['Country/Region'].value_counts().index
sns.countplot(data=covid, x='Country/Region', order=order_cv,
              palette='colorblind')
plt.title('국가별 관측 일수 (COVID 데이터)', fontsize=13, fontweight='bold')
plt.xlabel('국가'); plt.ylabel('관측 일수')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

---

## **3.2 막대 그래프 — 그룹별 통계량 비교**

```python
sns.barplot(data=df, x='group', y='value', estimator=np.mean)
```

`barplot`은 그룹별 **통계량**(기본은 평균)을 막대 높이로, **신뢰구간**(기본 95%)을 검은 선으로 표시한다.

### Ping 9 — 자치구별 평균 가격 막대그래프

In [ ]:
# 막대그래프 - 평균 vs 중앙값 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (A) 평균 가격 — 이상치에 민감
sns.barplot(data=airbnb, x='neighbourhood_group', y='price',
            estimator=np.mean, errorbar=('ci', 95),
            order=order_count, palette='colorblind', ax=axes[0])
axes[0].set_title('(A) 평균 가격 — 이상치에 민감', fontsize=12, fontweight='bold')
axes[0].set_xlabel('자치구'); axes[0].set_ylabel('평균 가격 ($)')

# (B) 중앙값 가격 — 이상치에 강건
sns.barplot(data=airbnb, x='neighbourhood_group', y='price',
            estimator=np.median, errorbar=('ci', 95),
            order=order_count, palette='colorblind', ax=axes[1])
axes[1].set_title('(B) 중앙값 가격 — 이상치에 강건', fontsize=12, fontweight='bold')
axes[1].set_xlabel('자치구'); axes[1].set_ylabel('중앙 가격 ($)')

plt.tight_layout(); plt.show()

### **결과 해석 — 평균 vs 중앙값**

- **평균**(왼쪽)은 이상치(고가 숙소)의 영향으로 **위쪽으로 끌어올려져** 있다.
- **중앙값**(오른쪽)은 이상치에 영향받지 않는다 — 자치구의 **전형적 가격**을 더 잘 대표한다.
- 둘의 차이가 클수록 분포가 비대칭(왜도가 큼)임을 시사한다.

> **실무 가이드**: 가격, 소득, 매출처럼 **오른쪽 꼬리가 긴 분포**에서는 평균보다 **중앙값**을 보고하는 것이 신뢰성이 높다.

### Pong 7 — 국가별 평균 일별 신규 확진자

학생 실습. 국가별 평균 일별 신규 확진자를 막대그래프로 비교하라.

In [ ]:
# Pong 7 - 국가별 평균 일별 신규 확진자
plt.figure(figsize=(11, 5))
order_n = (covid_clean.groupby('Country/Region')['NewCases']
           .mean().sort_values(ascending=False).index)
sns.barplot(data=covid_clean, x='Country/Region', y='NewCases',
            estimator=np.mean, order=order_n,
            palette='colorblind', errorbar=('ci', 95))
plt.title('국가별 평균 일별 신규 확진자 (95% CI)', fontsize=13, fontweight='bold')
plt.xlabel('국가'); plt.ylabel('평균 일별 신규 확진자')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

---

## **3.3 서브플롯 — EDA 한눈에 보기**

```python
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(..., ax=axes[0, 0])
sns.boxplot(..., ax=axes[0, 1])
...
plt.tight_layout()
```

여러 그래프를 **한 Figure에 격자로 배치**하여 EDA 결과를 한눈에 확인한다.

### Ping 10 — 2×2 EDA 대시보드

In [ ]:
# 2×2 EDA 대시보드
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) 가격 히스토그램
sns.histplot(data=airbnb, x='price', bins=50, kde=True, ax=axes[0, 0])
axes[0, 0].axvline(airbnb['price'].median(), color='red', ls='--', lw=2)
axes[0, 0].set_title('(A) 전체 가격 분포', fontsize=12, fontweight='bold')

# (B) 자치구별 박스플롯
sns.boxplot(data=under300, x='neighbourhood_group', y='price',
            order=order_count, palette='colorblind', ax=axes[0, 1])
axes[0, 1].set_title('(B) 자치구별 가격 ($300 이하)', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=15)

# (C) room_type 카운트
sns.countplot(data=airbnb, x='room_type', palette='colorblind', ax=axes[1, 0])
axes[1, 0].set_title('(C) 숙소 유형별 매물 수', fontsize=12, fontweight='bold')

# (D) 상관 히트맵
corr_d = airbnb[num_cols].corr()
sns.heatmap(corr_d, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1, 1], cbar_kws={'shrink':0.8}, annot_kws={'size':9})
axes[1, 1].set_title('(D) 상관 히트맵', fontsize=12, fontweight='bold')

plt.suptitle('Airbnb NYC EDA 대시보드', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

### **EDA 대시보드의 가치**

분포(A), 그룹 비교(B), 빈도(C), 관계(D)를 **한 페이지로** 보여주면 보고서·발표에서 데이터 전체상을 30초 안에 전달할 수 있다.

---

## **3.4 AI 연결 — EDA 5단계 체크리스트**

> **EDA는 ML 파이프라인의 필수 관문**이다. AutoML도 내부적으로 EDA를 수행한다. 다음 5가지를 그래프로 점검하지 않고 모델을 학습하면 "눈 감고 운전하는 것"과 같다.

| # | 점검 항목 | 도구 | 발견하는 문제 |
|---|---|---|---|
| 1 | 결측값 비율 | `isnull().sum()` 막대 | 삭제 vs 대체 결정 |
| 2 | 분포 형태 | 히스토그램 + KDE | 왜도, 이봉분포, 스케일링 필요성 |
| 3 | 이상치 존재 | 박스플롯 | 제거 vs 보존(이상 탐지) 결정 |
| 4 | 클래스 불균형 | 카운트플롯 | SMOTE·`class_weight` 전략 |
| 5 | 특성 간 관계 | 상관 히트맵 | 특성 선택, 다중공선성, **데이터 누수** |

In [ ]:
# EDA 5단계 체크리스트 — 한 번에 시각화
np.random.seed(42)
n = 500
df_eda = pd.DataFrame({
    'age':       np.concatenate([np.random.normal(35, 10, n-5),
                                 [150, -3, 200, np.nan, np.nan]]),  # 이상치 + 결측
    'income':    np.concatenate([np.random.exponential(30000, n-3),
                                 [np.nan, np.nan, np.nan]]),         # 결측 + 왜도
    'education': np.random.choice(['고졸','대졸','석사','박사'], n,
                                  p=[0.4, 0.35, 0.2, 0.05]),
    'purchased': np.concatenate([np.zeros(420), np.ones(80)]),       # 클래스 불균형
})

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. 결측값
ax = axes[0, 0]
missing = df_eda.isnull().sum()
colors_m = ['#ef4444' if v > 0 else '#10b981' for v in missing.values]
ax.bar(missing.index, missing.values, color=colors_m, alpha=0.7, edgecolor='white')
for i, v in enumerate(missing.values):
    if v > 0:
        ax.text(i, v + 0.5, f'{v}개', ha='center', fontsize=10, fontweight='bold', color='#ef4444')
ax.set_title('① 결측값 점검', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', labelsize=9)

# 2. 분포(왜도)
ax = axes[0, 1]
income_clean = df_eda['income'].dropna()
ax.hist(income_clean, bins=40, color='#f59e0b', alpha=0.6, edgecolor='white', density=True)
from scipy.stats import skew
sk = skew(income_clean)
ax.axvline(income_clean.mean(), color='red', ls='--', lw=2, label=f'평균={income_clean.mean():,.0f}')
ax.axvline(income_clean.median(), color='blue', ls=':', lw=2, label=f'중앙값={income_clean.median():,.0f}')
ax.set_title(f'② 분포 점검 — skew = {sk:.2f}', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# 3. 이상치
ax = axes[0, 2]
age_clean = df_eda['age'].dropna()
bp = ax.boxplot([age_clean], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3b82f6', alpha=0.4),
                flierprops=dict(marker='o', markerfacecolor='#ef4444',
                               markeredgecolor='#ef4444', markersize=8))
ax.set_title('③ 이상치 점검 — age', fontsize=12, fontweight='bold')
ax.text(0.95, 0.92, f'min={age_clean.min():.0f}\nmax={age_clean.max():.0f}\n→ 오류 의심',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, fontweight='bold', color='#ef4444',
        bbox=dict(facecolor='white', edgecolor='#ef4444', boxstyle='round,pad=0.3'))

# 4. 클래스 불균형
ax = axes[1, 0]
class_counts = df_eda['purchased'].value_counts()
ax.bar(['미구매(0)','구매(1)'], class_counts.values,
       color=['#3b82f6','#ef4444'], alpha=0.7, edgecolor='white')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 8, f'{v}\n({v/n*100:.0f}%)', ha='center', fontsize=10, fontweight='bold')
ax.set_title('④ 클래스 불균형 — 84:16', fontsize=12, fontweight='bold', color='#ef4444')

# 5. 상관관계
ax = axes[1, 1]
corr = df_eda[['age','income','purchased']].dropna().corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('⑤ 상관관계 점검', fontsize=12, fontweight='bold')

# 종합 판정
ax = axes[1, 2]
ax.axis('off')
ax.set_title('EDA 종합 판정', fontsize=12, fontweight='bold', color='#3b82f6')
checks = [
    ('① 결측값',   f'{missing.sum()}개 발견',           '#ef4444'),
    ('② 분포',     f'income skew={sk:.1f} → log 변환',  '#f59e0b'),
    ('③ 이상치',   'age에 -3, 150, 200 → 제거',         '#ef4444'),
    ('④ 불균형',   '84:16 → SMOTE 필요',                '#ef4444'),
    ('⑤ 상관',     '다중공선성 없음',                    '#10b981'),
]
for i, (name, desc, color) in enumerate(checks):
    y = 0.88 - i*0.16
    ax.text(0.05, y,    name, transform=ax.transAxes, fontsize=11, fontweight='bold', color=color)
    ax.text(0.05, y-0.06, f'  {desc}', transform=ax.transAxes, fontsize=9, color='#444')
ax.text(0.05, 0.04, '→ 전처리 후 모델 학습',
        transform=ax.transAxes, fontsize=11, fontweight='bold', color='#3b82f6')

plt.suptitle('EDA 5단계 체크리스트 — 모델 학습 전 필수 점검',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### **결과 해석**

이 한 장의 대시보드가 다음 의사결정을 만든다:

1. **결측값 5개** → `dropna()` vs `fillna()` 선택
2. **소득 skew = 2.5** → `np.log1p()` 적용 후 모델 입력
3. **age에 -3, 150, 200** → 명백한 입력 오류 → 제거
4. **84:16 불균형** → SMOTE 또는 `class_weight='balanced'`
5. **상관 점검 완료** → 다중공선성 없음 → 모든 특성 사용 가능

> **AI 연결**: Google AutoML, H2O.ai, Auto-sklearn 같은 자동 ML 도구도 내부적으로 결측값 → 분포 → 이상치 → 불균형 → 상관 순서로 점검한다. **EDA는 자동화된 AI 시대에도 사라지지 않는 기초 역량**이다.

### ✎ 개념 확인 7

EDA 5단계 체크리스트의 순서를 올바르게 정렬하시오.

(가) 클래스 불균형 점검 (나) 분포 형태 (다) 결측값 비율
(라) 특성 간 상관관계 (마) 이상치 존재

<details><summary>▶ 정답 보기</summary>

**(다) 결측값 → (나) 분포 → (마) 이상치 → (가) 불균형 → (라) 상관**

- 결측값을 먼저 처리해야 다른 분석이 정확하다.
- 분포를 보고 이상치 기준을 정한다.
- 이상치를 식별·처리한 뒤 클래스 불균형을 본다.
- 마지막으로 정제된 데이터에서 상관·다중공선성을 점검한다.
</details>

---

# **Part 4. 인터랙티브 · 고차원 시각화**(Interactive & High-Dimensional)

> **핵심 질문**: 정적 그래프로 부족할 때 어떻게 할 것인가?

| 도구 | 무엇을 할 수 있는가 | 대표 함수 |
|---|---|---|
| **Plotly Express** | 마우스 hover, 확대/축소, 회전 | `px.scatter`, `px.box`, `px.histogram` |
| **PCA**(주성분 분석) | 고차원 → 2D 압축, 전체 구조 | `sklearn.decomposition.PCA` |
| **t-SNE** | 군집(cluster) 시각화 특화 | `sklearn.manifold.TSNE` |

---

## **도구 선택 의사결정 트리**

```
질문: 무엇을 보고 싶은가?
│
├─ 🎯 정확한 값을 hover로 확인하고 싶다 ──→ Plotly
├─ 🌍 지도 위에 데이터를 표시하고 싶다  ──→ Plotly mapbox / Folium
├─ 📊 변수가 10개 이상이다              ──→ PCA → 2D 시각화
└─ 🔬 군집(cluster) 구조를 보고 싶다    ──→ t-SNE
```

## **4.1 Plotly Express — 인터랙티브의 표준**

```python
import plotly.express as px
fig = px.scatter(df, x='lon', y='lat', color='price',
                 hover_data=['name','room_type'],
                 color_continuous_scale='viridis')
fig.show()
```

Plotly의 4가지 강점:
1. **`hover`**: 마우스를 올리면 해당 점의 모든 정보가 표시된다.
2. **`drag`**: 영역을 드래그하여 확대할 수 있다.
3. **`save`**: 카메라 아이콘으로 PNG 저장이 가능하다.
4. **`legend`**: 범례 클릭으로 그룹을 켜고 끌 수 있다.

### Ping 11 — Airbnb 위치 × 가격 인터랙티브 지도

In [ ]:
# Plotly Express 인터랙티브 산점도
import plotly.express as px

# 표본 추출 (성능)
sample = airbnb.sample(min(3000, len(airbnb)), random_state=42)

fig = px.scatter(
    sample,
    x='longitude', y='latitude',
    color='price',
    color_continuous_scale='viridis',  # ★ 색맹 안전 연속 컬러맵
    hover_data={'neighbourhood_group': True,
                'room_type': True,
                'price': ':.0f',
                'longitude': ':.3f',
                'latitude': ':.3f'},
    opacity=0.5,
    title='NYC Airbnb 위치 × 가격 (인터랙티브) — 마우스를 올려보라',
    labels={'longitude':'경도', 'latitude':'위도', 'price':'가격($)'},
)
fig.update_layout(width=800, height=650)
fig.show()

### **인터랙티브 탐색 가이드**

1. 마우스를 점에 올려 — 해당 숙소의 자치구, 유형, 가격을 확인한다.
2. 노란색(고가) 점이 밀집한 지역을 드래그로 확대한다.
3. 더블클릭으로 원래 보기로 복원한다.
4. 우측 상단의 카메라 아이콘으로 PNG 저장이 가능하다.

> **실무 가이드**: 정적 그래프(Matplotlib/Seaborn)는 **보고서·논문**에, 인터랙티브 그래프(Plotly)는 **대시보드·발표**에 적합하다. 두 도구는 경쟁이 아니라 **상호 보완**이다.

### ✎ 개념 확인 8

다음 상황에서 정적 vs 인터랙티브 시각화 선택으로 가장 적절한 것은?

| 상황 | 선택 |
|---|---|
| (가) 학회 논문에 그림으로 첨부 | ____ |
| (나) 임원 발표용 대시보드 | ____ |
| (다) 데이터셋 1억 행 시각화 | ____ |

<details><summary>▶ 정답 보기</summary>

| 상황 | 선택 | 이유 |
|---|---|---|
| (가) 논문 | **정적**(Matplotlib) | PDF·인쇄에 안정 |
| (나) 발표 | **인터랙티브**(Plotly) | hover로 청중 질문에 즉답 |
| (다) 1억 행 | **정적 + 다운샘플링** | Plotly는 10만 행 이상에서 매우 느려짐. 표본 추출 후 시각화 |
</details>

### Pong 8 — COVID 시계열 인터랙티브 라인 플롯

학생 실습. 국가별 누적 확진자를 시계열로 그려라.

In [ ]:
# Pong 8 - COVID 시계열 라인 플롯
fig = px.line(
    covid,
    x='Date', y='Confirmed', color='Country/Region',
    title='COVID-19 — 국가별 누적 확진자 시계열 (인터랙티브)',
    labels={'Date':'날짜', 'Confirmed':'누적 확진자',
            'Country/Region':'국가'},
)
fig.update_layout(width=900, height=500, hovermode='x unified')
fig.show()

---

## **4.2 PCA — 고차원 데이터의 2D 압축**

> 우리 눈은 2D/3D만 볼 수 있다. 그러나 실제 데이터는 수십~수백 차원이다. **차원 축소**(Dimensionality Reduction)는 고차원 데이터를 보존하면서 2D로 압축한다.

| 기법 | 원리 | 강점 | 약점 |
|---|---|---|---|
| **PCA** | 분산이 최대인 축으로 투영 | 빠르다, 해석 가능 | 비선형 구조 못 잡음 |
| **t-SNE** | 이웃 관계 보존하며 축소 | 군집이 명확히 보임 | 느리다, 거리 의미 없음 |

### Ping 12 — 손글씨 숫자(64차원)를 2D로

In [ ]:
# PCA — 손글씨 숫자 64차원을 2차원으로
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)  # 64차원
y_digits = digits.target

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digits)

plt.figure(figsize=(11, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits,
                      cmap='tab10', s=20, alpha=0.6, edgecolors='none')
plt.colorbar(scatter, label='숫자 (0~9)')
plt.title(f'PCA: 64차원 → 2차원 (설명 분산: {pca.explained_variance_ratio_.sum():.1%})',
          fontsize=13, fontweight='bold')
plt.xlabel('PC1 (제1주성분)'); plt.ylabel('PC2 (제2주성분)')
plt.show()

print(f'\n원본 차원: {X_digits.shape[1]}')
print(f'압축 후 차원: {X_pca.shape[1]}')
print(f'설명 분산: PC1={pca.explained_variance_ratio_[0]:.3f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.3f}')
print(f'합계: {pca.explained_variance_ratio_.sum():.3f}')

### **결과 해석**

- **PC1, PC2**는 원본 64차원 데이터의 분산을 가장 많이 설명하는 두 축이다.
- **설명 분산**(explained variance) 합이 약 21~30% — PCA가 주요 구조를 일부 잡지만, 비선형 군집은 흐릿하게 섞여 있다.
- 빠르고 해석 가능하지만 **군집 분리는 약하다**.

## **4.3 t-SNE — 군집 시각화의 표준**

```python
from sklearn.manifold import TSNE
X_tsne = TSNE(n_components=2, perplexity=30).fit_transform(X)
```

t-SNE는 **이웃 관계를 보존**하면서 차원을 축소한다. PCA와 같은 데이터에 적용하면 **군집이 훨씬 명확히 분리**되어 보인다.

### Ping 13 — 같은 데이터를 t-SNE로

In [ ]:
# t-SNE - 같은 64차원 데이터를 2차원으로 (시간이 다소 걸림)
from sklearn.manifold import TSNE

# 표본 추출 (속도)
np.random.seed(42)
idx = np.random.choice(len(X_digits), 1000, replace=False)
X_sub, y_sub = X_digits[idx], y_digits[idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# (A) PCA 결과 - 비교
pca_sub = PCA(n_components=2).fit_transform(X_sub)
sc1 = axes[0].scatter(pca_sub[:, 0], pca_sub[:, 1], c=y_sub,
                       cmap='tab10', s=20, alpha=0.6)
axes[0].set_title('(A) PCA — 군집이 흐릿하게 섞임', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0])

# (B) t-SNE 결과
sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub,
                       cmap='tab10', s=20, alpha=0.6)
axes[1].set_title('(B) t-SNE — 같은 숫자끼리 명확히 모임!', fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
plt.colorbar(sc2, ax=axes[1])

plt.tight_layout(); plt.show()

### **결과 해석 — PCA vs t-SNE**

- **PCA**(왼쪽): 전체 분산을 본다 → 글로벌 구조 파악, 빠름.
- **t-SNE**(오른쪽): 가까운 이웃 관계를 본다 → **군집(cluster) 구조가 명확히 분리**된다.

> **AI 연결 — 임베딩 시각화**:
> - GPT의 토큰 임베딩(768~16384차원)을 t-SNE로 시각화하면 **의미가 비슷한 단어들이 가까이 모여** 있는 것을 확인할 수 있다.
> - **`king - man + woman ≈ queen`** 같은 임베딩 산술 관계도 2D에서 눈으로 확인 가능하다.
> - Word2Vec, BERT, Sentence-BERT 임베딩의 **품질 점검 표준 도구**가 t-SNE 시각화이다.

> **주의사항**: t-SNE 결과의 **거리는 절대적 의미를 갖지 않는다**. 군집의 분리·이웃 관계만 신뢰하라. `perplexity` 값에 따라 결과가 크게 달라진다(보통 5~50).

### ✎ 개념 확인 9

다음 상황에서 PCA와 t-SNE 중 어느 것을 선택해야 하는가?

| 상황 | 선택 |
|---|---|
| (가) 1000개 변수를 빠르게 50개로 압축한 뒤 모델에 넣고 싶다 | ____ |
| (나) 고객 100만 명을 군집화한 결과를 시각화하고 싶다 | ____ |
| (다) 단어 임베딩에서 의미적으로 비슷한 단어들이 가까이 있는지 보고 싶다 | ____ |

<details><summary>▶ 정답 보기</summary>

| 상황 | 선택 | 이유 |
|---|---|---|
| (가) 50차원 압축 | **PCA** | 빠르고 명확한 변환 행렬을 가짐. 다른 데이터에도 동일하게 적용 가능 |
| (나) 100만 군집 시각화 | **t-SNE 또는 UMAP** | 군집 분리에 탁월. 단, 100만은 표본 추출 권장 |
| (다) 임베딩 시각화 | **t-SNE** | 이웃 관계 보존이 핵심. 의미 유사성을 거리로 표현 |
</details>

### Pong 9 — Iris 데이터를 PCA로 2D 시각화

학생 실습. 4차원 Iris 데이터를 PCA로 2D로 압축하여 시각화하라.

In [ ]:
# Pong 9 - Iris 4차원 → 2차원
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

pca_iris = PCA(n_components=2, random_state=42)
X_iris_pca = pca_iris.fit_transform(X_iris)

plt.figure(figsize=(9, 7))
for i, name in enumerate(iris.target_names):
    mask = y_iris == i
    plt.scatter(X_iris_pca[mask, 0], X_iris_pca[mask, 1],
                label=name, s=60, alpha=0.7,
                edgecolors='white', lw=0.5)
plt.title(f'Iris 데이터 PCA — 4차원 → 2차원 '
          f'(설명 분산 {pca_iris.explained_variance_ratio_.sum():.1%})',
          fontsize=13, fontweight='bold')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='품종')
plt.show()

---

## **4.4 시각화의 미래 — 설명 가능한 AI**(XAI)

EDA는 학습 **전**의 시각화이다. 학습 **후**에는 모델이 **왜 그런 예측을 했는지** 시각화해야 한다 — 이것이 **XAI**(Explainable AI)이다.

| 시점 | 시각화 도구 | 목적 |
|---|---|---|
| **학습 전 (EDA)** | 히스토그램, 박스플롯, 상관 히트맵 | 데이터 품질 점검 |
| **학습 중** | 학습 곡선(Learning Curve), 손실 그래프 | 과적합 진단 |
| **학습 후 (XAI)** | 특성 중요도, SHAP, Grad-CAM | 모델 예측 근거 설명 |

> **EU AI Act**(2024)는 고위험 AI 시스템에 대해 **"설명 가능성"을 법적으로 요구**한다. 의료·금융·법률 AI는 왜 그런 판단을 했는지 설명할 수 있어야 한다. 시각화는 이 설명의 핵심 도구이다.

In [ ]:
# XAI 맛보기 - 특성 중요도 시각화
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_iris, y_iris)

importances = pd.Series(rf.feature_importances_,
                        index=iris.feature_names).sort_values()

plt.figure(figsize=(10, 5))
colors_imp = ['#3b82f6' if v < 0.2 else '#f59e0b' if v < 0.4 else '#10b981'
              for v in importances.values]
plt.barh(importances.index, importances.values, color=colors_imp,
         alpha=0.8, edgecolor='white')
for i, (name, val) in enumerate(importances.items()):
    plt.text(val + 0.005, i, f'{val:.3f}', va='center',
             fontsize=11, fontweight='bold')
plt.title('Iris 분류기 — 특성 중요도 (XAI 맛보기)', fontsize=13, fontweight='bold')
plt.xlabel('중요도')
plt.tight_layout(); plt.show()

print('\n해석:')
print('  petal length, petal width가 가장 중요한 특성이다.')
print('  → "왜 이 꽃을 versicolor로 분류했는가?"의 설명 근거가 된다.')

### **결과 해석**

- **petal length, petal width**가 분류에 결정적 기여를 한다.
- 모델이 어떤 특성을 보고 판단했는지 **시각적으로 설명**할 수 있다 — 이것이 XAI의 출발점이다.

> 본 차시에서 배운 시각화 기술은 **ML 파이프라인 전체**(학습 전 → 학습 중 → 학습 후)에 관통한다. 9~10주차의 학습 곡선, 13~14주차의 SHAP/Grad-CAM도 모두 같은 시각화 원리 위에 서 있다.

---

## **Part 3 + Part 4 연습문제** — 비교·인터랙티브·고차원 10문항

---

### 문제 1. 카운트플롯 활용

`countplot`이 가장 먼저 발견해야 할 문제는?

<details><summary>▶ 정답</summary>**클래스 불균형**(class imbalance). 이진 분류에서 95:5 같은 비율은 모델 학습 전에 반드시 발견해 SMOTE·class_weight 등으로 처리해야 한다.</details>

---

### 문제 2. 평균 vs 중앙값 막대그래프

가격·소득·매출처럼 오른쪽 꼬리가 긴 데이터에서 막대그래프는 어떤 통계량을 사용해야 하는가?

<details><summary>▶ 정답</summary>**중앙값**(median). 이상치에 강건하기 때문이다. `sns.barplot(..., estimator=np.median)`으로 변경한다.</details>

---

### 문제 3. 서브플롯 격자

`fig, axes = plt.subplots(2, 3)`로 만든 axes의 모양은?

<details><summary>▶ 정답</summary>2행 × 3열의 2차원 numpy 배열. `axes[0,0]`, `axes[1,2]` 등으로 접근한다.</details>

---

### 문제 4. EDA 5단계 순서

EDA 5단계 체크리스트의 올바른 순서는?

<details><summary>▶ 정답</summary>**결측값 → 분포 → 이상치 → 불균형 → 상관**. 결측을 먼저 처리해야 다른 분석이 정확하고, 정제된 데이터에서 마지막에 상관·다중공선성을 점검한다.</details>

---

### 문제 5. Plotly 사용 적합도

다음 중 Plotly가 가장 빛나는 상황은?

**(A)** 논문 PDF에 첨부할 흑백 그림 (B) 임원 보고용 대시보드 (C) 1억 행 데이터 시각화

<details><summary>▶ 정답</summary>**(B) 대시보드**. hover로 즉시 정보 확인이 가능하여 발표 도중 청중의 질문에 즉답할 수 있다. (A)는 정적, (C)는 표본 추출 후에야 가능하다.</details>

---

### 문제 6. PCA 설명 분산

`PCA(n_components=2).fit(X)` 후 `pca.explained_variance_ratio_.sum() = 0.85`는 무엇을 의미하는가?

<details><summary>▶ 정답</summary>2개의 주성분이 원본 데이터 **전체 분산의 85%를 설명**한다는 의미. 차원 축소가 효과적이며 정보 손실이 15% 수준이라는 것이다.</details>

---

### 문제 7. t-SNE 거리 해석

t-SNE 결과 그림에서 두 군집이 멀리 떨어져 있다. 이 거리의 의미는?

<details><summary>▶ 정답</summary>**거리의 절대값은 의미가 없다**. t-SNE는 이웃 관계만 보존하므로 군집 간 분리만 신뢰해야 한다. "A 군집이 C보다 B에 가깝다"는 해석도 위험하다. 군집의 **존재**만 신뢰하라.</details>

---

### 문제 8. PCA vs t-SNE 선택

10000개 변수의 신호를 모델 입력 전 200차원으로 압축하고 싶다. 어느 것을 사용?

<details><summary>▶ 정답</summary>**PCA**. 명시적 변환 행렬을 갖기 때문에 **새 데이터에도 동일하게 적용**할 수 있다. t-SNE는 시각화 전용이며 새 데이터에 일반화할 수 없다.</details>

---

### 문제 9. 코드 빈칸 — Plotly

```python
# 빈칸: 자치구별 가격 박스플롯, 색상별 분리
fig = px._______(airbnb,
                 x='neighbourhood_group', y='price',
                 color='_______',
                 title='지역별 가격')
fig.show()
```

<details><summary>▶ 정답</summary>

```python
fig = px.box(airbnb,
             x='neighbourhood_group', y='price',
             color='neighbourhood_group',
             title='지역별 가격')
fig.show()
```
</details>

---

### 문제 10. AI 연결 — 임베딩 시각화

GPT나 BERT의 768차원 단어 임베딩의 **품질을 점검**하려면 어떤 시각화가 표준인가?

<details><summary>▶ 정답</summary>**t-SNE 또는 UMAP 시각화**. 의미가 비슷한 단어들이 2D 평면에서 가까이 모여 있는지 확인한다. 예: "king, queen, prince"가 한 군집을 형성하면 임베딩이 **의미를 잘 학습**한 것이다.</details>

---

> **Part 3 + 4 마무리**: 비교·인터랙티브·고차원 시각화는 분석가의 **상황 적응력**을 결정한다. 정적 그래프로 충분한 상황과 인터랙티브가 필요한 상황, 2D로 충분한 데이터와 차원 축소가 필요한 데이터를 구분할 수 있어야 진짜 분석가이다.

---

# **2차시 마무리**

## **그래프 선택 결정 트리** — 이 한 장이 본 차시의 결론이다

```
        ┌──────────────────────────────────────────┐
        │  내가 보고 싶은 것은 무엇인가?               │
        └────────────────┬─────────────────────────┘
                         │
        ┌────────────────┼────────────────────────┐
        │                │                        │
   ┌────▼────┐     ┌─────▼────┐            ┌──────▼────┐
   │ 한 변수  │     │ 두 변수  │            │ 그룹 비교 │
   │ 분포    │     │ 관계    │            │           │
   └─────────┘     └──────────┘            └───────────┘
        │                │                        │
   ┌────┴────┐     ┌─────┴─────┐          ┌──────┴────┐
   │히스토그램│     │  산점도    │          │ 막대그래프 │
   │ + KDE   │     │           │          │           │
   │박스플롯  │     │  회귀선    │          │ 카운트플롯 │
   │바이올린  │     │  pairplot │          │ 그룹 박스  │
   └─────────┘     │  히트맵    │          └───────────┘
                   └───────────┘

   ┌──────────────────────────────────────┐
   │  변수가 10개 이상이면 → PCA, t-SNE     │
   │  발표·대시보드 → Plotly 인터랙티브     │
   │  지도 데이터 → Folium 또는 Plotly map  │
   └──────────────────────────────────────┘
```

## **핵심 정리표**

| 영역 | 1차 선택 | 비교·심화 | AI 연결 |
|---|---|---|---|
| **분포** | `histplot` + KDE | `boxplot`·`violinplot` | 정규분포 가정·Layer Norm |
| **관계** | `scatterplot` | `regplot`·`pairplot`·`heatmap` | 특성 선택·다중공선성·**데이터 누수** |
| **비교** | `countplot` | `barplot`·서브플롯 | 클래스 불균형·SMOTE |
| **고차원** | `PCA` | `t-SNE`·`UMAP` | 임베딩 품질 점검·XAI |

## **3가지 황금 규칙**

1. **항상 그래프를 먼저 그려라** — 앤스콤의 사중주가 증명한 EDA 제1원칙.
2. **`sns.set_palette("colorblind")`** — 모든 노트북 첫 셀에 두는 습관.
3. **상관 ≠ 인과** — "~와 관련이 있다"(상관) vs "~때문이다"(인과)를 엄격히 구분.

## **이번 차시에서 배운 것**

- 분포·관계·비교·고차원의 **4영역 시각화 도구**
- **색맹 안전 팔레트**(colorblind, viridis, cividis) 사용 습관
- **EDA 5단계 체크리스트** — 결측·분포·이상치·불균형·상관
- **AI 연결**: 분포 정규화, 데이터 누수, 클래스 불균형, 임베딩 시각화, XAI

## **다음 차시 예고**

분석은 시각으로 이해되고, 모델로 일반화된다. 다음 차시에서는 본 차시에서 발견한 패턴을 **수치적으로 일반화하는 머신러닝 워크플로우**(데이터 분할 → 전처리 → 학습 → 평가)를 다룬다.

---

> **수업 종료. 수고하셨습니다.**